# 03 — Cluster Profiling

**Purpose:** Turn cluster numbers into business stories.  
**Questions we answer here:**
- What defines each segment? What are their characteristics?
- Was k=5 the right choice? (Elbow method validation)
- Where is the revenue concentrated? What's at risk?
- What actions should we recommend per segment?

**Tables used:** `marts.mart_cluster_output`, `marts.mart_cluster_profiles`, `marts.mart_cluster_summary`

In [ ]:
from google.cloud import bigquery
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

PROJECT = 'adg-internal-tech-sandbox'
client = bigquery.Client(project=PROJECT)

def q(sql):
    return client.query(sql).to_dataframe()

print(f'Connected to {PROJECT}')

## 1. Model evaluation — Davies-Bouldin index
Lower is better. Under 2.0 is decent for business use.

In [ ]:
eval_result = q(f"""
    SELECT * FROM ML.EVALUATE(MODEL `{PROJECT}.analytics.kmeans_customer_segments`)
""")
eval_result

## 2. Training convergence
Loss should decrease and flatten — that means the model found stable cluster centers.

In [ ]:
training = q(f"""
    SELECT iteration,
           ROUND(loss, 4) AS loss,
           ROUND(loss - LAG(loss) OVER (ORDER BY iteration), 4) AS improvement
    FROM ML.TRAINING_INFO(MODEL `{PROJECT}.analytics.kmeans_customer_segments`)
    ORDER BY iteration
""")

fig = px.line(training, x='iteration', y='loss',
              title='Training convergence — loss should flatten',
              markers=True)
fig.show()
training

## 3. Elbow method — validating k=5
Train models for k=3 through k=8 and compare Davies-Bouldin scores.

**Note:** This section takes ~5 minutes to run (6 model trainings). Skip if already validated.

In [ ]:
# Uncomment to run the full elbow analysis.
# Each CREATE MODEL takes ~60 seconds.

FEATURE_SELECT = f"""
    SELECT val_trns, nr_trns, lst_trns_days, avg_val,
           active_destinations, active_nav_categories,
           NR_TRNS_WEEKEND, NR_TRNS_WEEK, active_months
    FROM `{PROJECT}.analytics.int_rfm_scores`
"""

results = []
for k in [3, 4, 5, 6, 7, 8]:
    model_name = f'{PROJECT}.sandbox.elbow_k{k}' if k != 5 else f'{PROJECT}.analytics.kmeans_customer_segments'
    
    if k != 5:  # k=5 model already exists
        print(f'Training k={k}...')
        client.query(f"""
            CREATE OR REPLACE MODEL `{model_name}`
            OPTIONS (model_type='KMEANS', num_clusters={k},
                     standardize_features=TRUE, max_iterations=20)
            AS {FEATURE_SELECT}
        """).result()
    
    ev = q(f"SELECT {k} AS k, * FROM ML.EVALUATE(MODEL `{model_name}`)")
    results.append(ev)
    print(f'  k={k}: Davies-Bouldin = {ev.iloc[0]["davies_bouldin_index"]:.4f}')

elbow = pd.concat(results)
elbow

In [ ]:
# Visualize elbow
fig = go.Figure()
fig.add_trace(go.Scatter(x=elbow['k'], y=elbow['davies_bouldin_index'],
                         mode='lines+markers', name='Davies-Bouldin',
                         line=dict(color='#2E75B6')))
fig.add_vline(x=5, line_dash='dash', line_color='red',
              annotation_text='k=5 (selected)')
fig.update_layout(title='Elbow method — Davies-Bouldin index by k',
                  xaxis_title='Number of clusters (k)',
                  yaxis_title='Davies-Bouldin index (lower = better)')
fig.show()

In [ ]:
# Cleanup sandbox models
for k in [3, 4, 6, 7, 8]:
    client.query(f'DROP MODEL IF EXISTS `{PROJECT}.sandbox.elbow_k{k}`').result()
    print(f'Dropped elbow_k{k}')

## 4. Segment overview
What does each segment look like?

In [ ]:
summary = q(f"""
    SELECT segment_name, customer_count, pct_of_total,
           avg_total_spend, avg_transactions, avg_recency_days,
           business_description, recommended_action
    FROM `{PROJECT}.marts.mart_cluster_summary`
    ORDER BY avg_total_spend DESC
""")
summary

## 5. Cluster centroids — what defines each group?
The centroid values show the 'average' customer in each cluster.

In [ ]:
centroids = q(f"""
    SELECT centroid_id, feature, ROUND(numerical_value, 2) AS value
    FROM ML.CENTROIDS(MODEL `{PROJECT}.analytics.kmeans_customer_segments`)
    ORDER BY centroid_id, feature
""")

# Pivot for readability
centroid_pivot = centroids.pivot(index='feature', columns='centroid_id', values='value')
centroid_pivot

## 6. Indexed radar chart
Each metric indexed to 0-100 (where 100 = highest segment). Shows relative strengths.

In [ ]:
radar_data = q(f"""
    SELECT segment_name,
        ROUND(avg_total_spend / MAX(avg_total_spend) OVER() * 100, 0) AS spend_index,
        ROUND(avg_transactions / MAX(avg_transactions) OVER() * 100, 0) AS frequency_index,
        ROUND((1 - avg_recency_days / MAX(avg_recency_days) OVER()) * 100, 0) AS recency_index,
        ROUND(avg_merchants / MAX(avg_merchants) OVER() * 100, 0) AS diversity_index,
        ROUND(avg_active_months / MAX(avg_active_months) OVER() * 100, 0) AS consistency_index
    FROM `{PROJECT}.marts.mart_cluster_profiles`
""")

categories = ['spend_index', 'frequency_index', 'recency_index',
              'diversity_index', 'consistency_index']

fig = go.Figure()
for _, row in radar_data.iterrows():
    fig.add_trace(go.Scatterpolar(
        r=[row[c] for c in categories] + [row[categories[0]]],
        theta=['Spend', 'Frequency', 'Recency', 'Diversity', 'Consistency', 'Spend'],
        name=row['segment_name'],
        fill='toself', opacity=0.5
    ))

fig.update_layout(title='Segment radar chart (indexed 0-100)',
                  polar=dict(radialaxis=dict(range=[0, 100])),
                  height=500)
fig.show()

## 7. Revenue concentration
"Champions are X% of customers but Y% of revenue" — the pitch headline.

In [ ]:
revenue = q(f"""
    SELECT segment_name,
           COUNT(*) AS customers,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct_customers,
           ROUND(SUM(val_trns), 0) AS total_spend,
           ROUND(SUM(val_trns) * 100.0 / SUM(SUM(val_trns)) OVER(), 1) AS pct_revenue
    FROM `{PROJECT}.marts.mart_cluster_output`
    GROUP BY segment_name
    ORDER BY total_spend DESC
""")

fig = go.Figure()
fig.add_trace(go.Bar(name='% of customers', x=revenue['segment_name'],
                     y=revenue['pct_customers'], marker_color='#607D8B'))
fig.add_trace(go.Bar(name='% of revenue', x=revenue['segment_name'],
                     y=revenue['pct_revenue'], marker_color='#2E75B6'))
fig.update_layout(barmode='group', title='Revenue concentration by segment',
                  yaxis_title='%')
fig.show()

# The pitch headline
champ = revenue[revenue['segment_name'] == 'Champions'].iloc[0]
print(f"\nPitch headline: Champions are {champ['pct_customers']}% of customers "
      f"but drive {champ['pct_revenue']}% of revenue.")

## 8. Win-back opportunity
At Risk + Dormant customers with high monetary scores = best re-engagement targets.

In [ ]:
winback = q(f"""
    SELECT segment_name,
           COUNT(*) AS customers,
           ROUND(AVG(val_trns), 0) AS avg_spend,
           COUNTIF(m_score >= 4) AS high_monetary,
           ROUND(COUNTIF(m_score >= 4) * 100.0 / COUNT(*), 1) AS pct_high_monetary,
           ROUND(SUM(CASE WHEN m_score >= 4 THEN val_trns ELSE 0 END), 0) AS high_monetary_spend
    FROM `{PROJECT}.marts.mart_cluster_output`
    WHERE segment_name IN ('At Risk', 'Dormant')
    GROUP BY segment_name
""")

print('Win-back targets: high-value customers who are slipping away')
winback

## 9. Demographics per segment
Who are the Champions? Who are At Risk?

In [ ]:
demo_seg = q(f"""
    SELECT segment_name, customer_count, avg_age, avg_income,
           age_18_25, age_26_35, age_36_45, age_46_60, age_over_60,
           male_count, female_count, top_age_group, top_income_group
    FROM `{PROJECT}.marts.mart_cluster_profiles`
    ORDER BY avg_total_spend DESC
""")
demo_seg

In [ ]:
# Age distribution stacked by segment
age_cols = ['age_18_25', 'age_26_35', 'age_36_45', 'age_46_60', 'age_over_60']
age_labels = ['18-25', '26-35', '36-45', '46-60', '60+']

age_melted = demo_seg[['segment_name'] + age_cols].melt(
    id_vars='segment_name', var_name='age_group', value_name='count'
)
age_melted['age_group'] = age_melted['age_group'].map(dict(zip(age_cols, age_labels)))

fig = px.bar(age_melted, x='segment_name', y='count', color='age_group',
             title='Age distribution per segment', barmode='stack')
fig.show()